# Import & Config

In [1]:
!wget https://www.lamsade.dauphine.fr/~cazenave/project2026.zip
!unzip project2026.zip
!ls -l

--2026-03-29 15:38:00--  https://www.lamsade.dauphine.fr/~cazenave/project2026.zip
Resolving www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)... 193.48.71.250
Connecting to www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)|193.48.71.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 138578548 (132M) [application/zip]
Saving to: ‘project2026.zip’

project2026.zip     100%[===================>] 132.16M  14.7MB/s    in 9.7s    

2026-03-29 15:38:11 (13.7 MB/s) - ‘project2026.zip’ saved [138578548/138578548]

Archive:  project2026.zip
  inflating: games.data              
  inflating: golois.cpython-312-x86_64-linux-gnu.so  
total 665484
-rw-r--r-- 1 root root 542497580 Oct  7  2022 games.data
-rwxr-xr-x 1 root root    284672 Oct  1 15:09 golois.cpython-312-x86_64-linux-gnu.so
---------- 1 root root     88144 Mar 29 15:38 __notebook__.ipynb
-rw-r--r-- 1 root root 138578548 Oct  1 20:02 project2026.zip


In [2]:
!pip install wandb weave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 955.0/955.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 119.2 MB/s eta 0:00:00


In [3]:
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_KEY = UserSecretsClient().get_secret("monitor GO")
wandb.login(key=WANDB_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rouibiamine7 (rouibiamine7-universit-paris-dauphine-psl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# BASE CODE (UNCHANGED)

In [4]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE BASE — Infrastructure partagée | Projet Go 19×19
#  À exécuter UNE SEULE FOIS avant n'importe quel modèle
# ═══════════════════════════════════════════════════════════════════

import os
import gc
import csv
import random
import shutil
import datetime
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers, regularizers
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Juste après les imports ───────────────────────────────────────

# ← AJOUTÉ : mixed precision — FP16 sur Tensor Cores (T4 / P100)
keras.mixed_precision.set_global_policy('mixed_float16')

# ← AJOUTÉ : détection multi-GPU
strategy = tf.distribute.MirroredStrategy()
NUM_GPUS  = strategy.num_replicas_in_sync
print(f"GPUs détectés : {NUM_GPUS}")


import golois

# ───────────────────────────────────────────────────────────────────
# CONSTANTES GLOBALES
# ───────────────────────────────────────────────────────────────────
DRIVE_DIR  = '/kaggle/working/go_project3'
RUNS_DIR   = 'runs'
PLANES     = 31
MOVES      = 361
N          = 10_000

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(RUNS_DIR,  exist_ok=True)

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs       : {tf.config.list_physical_devices('GPU')}")

# ───────────────────────────────────────────────────────────────────
# SEEDS
# ───────────────────────────────────────────────────────────────────
def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED']       = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seeds(42)
# ───────────────────────────────────────────────────────────────────
# DONNÉES — init une seule fois, réutilisées par tous les modèles
# ───────────────────────────────────────────────────────────────────
print("\nInitialisation des données...", flush=True)

# Train (écrasé à chaque getBatch)
input_data = np.random.randint(2, size=(N, 19, 19, PLANES)).astype('float32')
policy     = keras.utils.to_categorical(
                np.random.randint(MOVES, size=(N,)), num_classes=MOVES
             ).astype('float32')
value      = np.random.randint(2, size=(N,)).astype('float32')
end        = np.random.randint(2, size=(N, 19, 19, 2)).astype('float32')
groups     = np.zeros((N, 19, 19, 1), dtype='float32')

# Val set fixe — rempli une seule fois, jamais retouché
val_input  = np.zeros((N, 19, 19, PLANES), dtype='float32')
val_policy = np.zeros((N, MOVES),          dtype='float32')
val_value  = np.zeros((N,),                dtype='float32')
val_end    = np.zeros((N, 19, 19, 2),      dtype='float32')

golois.getValidation(val_input, val_policy, val_value, val_end)
print("✅ Val set fixe chargé — ne sera plus modifié\n")

# ───────────────────────────────────────────────────────────────────
# HELPERS
# ───────────────────────────────────────────────────────────────────
def make_run_dir(config):
    """Crée le dossier de run horodaté et écrit config.txt."""
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name  = (f"model{config['model_id']}_{config['model_name']}"
                 f"_lr{config['lr']}_seed{config['seed']}_{timestamp}")
    run_dir   = os.path.join(RUNS_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    with open(os.path.join(run_dir, 'config.txt'), 'w') as f:
        for k, v in config.items():
            f.write(f"{k} = {v}\n")
        f.write(f"\ntf_version = {tf.__version__}\n")
        f.write(f"gpu        = {tf.config.list_physical_devices('GPU')}\n")
        f.write(f"timestamp  = {timestamp}\n")

    print(f"\n{'='*60}")
    print(f"  RUN : {run_name}")
    print(f"  DIR : {run_dir}")
    print(f"{'='*60}\n")
    return run_dir, run_name, timestamp


def make_csv(run_dir):
    """Crée les fichiers CSV train et val avec leurs headers."""
    csv_fields = ['epoch', 'loss', 'policy_loss', 'value_loss',
                  'policy_categorical_accuracy', 'value_mae']
    val_fields = ['epoch', 'total_loss', 'policy_loss', 'value_loss',
                  'policy_categorical_accuracy', 'value_mae']

    csv_path = os.path.join(run_dir, 'history.csv')
    val_path = os.path.join(run_dir, 'val_results.csv')

    with open(csv_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=csv_fields).writeheader()
    with open(val_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=val_fields).writeheader()

    return csv_path, csv_fields, val_path, val_fields


def plot_curves(history_log, run_dir, run_name, epoch):
    """Sauvegarde les courbes train intermédiaires."""
    epochs_range = history_log['epoch']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(run_name, fontsize=9)

    axes[0].plot(epochs_range, history_log['loss'], 'b-o', markersize=3)
    axes[0].set_title('Total Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history_log['policy_categorical_accuracy'], 'g-o', markersize=3)
    axes[1].set_title('Policy Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs_range, history_log['value_mae'], 'r-o', markersize=3)
    axes[2].set_title('Value MAE'); axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(run_dir, f'curves_epoch{epoch}.png')
    plt.savefig(plot_path, dpi=120)
    plt.close()
    print(f"  [PLOT] → {plot_path}")


def save_final_plots(history_log, run_dir):
    """Sauvegarde les graphiques rapport final."""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    epochs = history_log['epoch']

    ax1.plot(epochs, history_log['policy_categorical_accuracy'],
             color='#2ca02c', label='Policy Accuracy')
    ax1.set_title('Évolution de la précision (Policy)')
    ax1.set_xlabel('Époques'); ax1.set_ylabel('Précision'); ax1.legend()

    ax2.plot(epochs, history_log['loss'], color='#d62728', label='Total Loss')
    ax2.set_title('Évolution de la perte (Loss)')
    ax2.set_xlabel('Époques'); ax2.set_ylabel('Perte'); ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'report_visuals.png'), dpi=300)
    plt.close()
    print("✅ report_visuals.png sauvegardé")


def drive_sync_light(run_dir, run_name):
    """Sync léger en cours de run : CSV + PNG + TXT uniquement."""
    drive_run_dir = os.path.join(DRIVE_DIR, run_name)
    os.makedirs(drive_run_dir, exist_ok=True)
    for fname in os.listdir(run_dir):
        if fname.endswith(('.csv', '.png', '.txt')):
            shutil.copy(os.path.join(run_dir, fname),
                        os.path.join(drive_run_dir, fname))
    print(f"  [DRIVE] Sync léger → {drive_run_dir}")


def drive_sync_full(run_dir, run_name):
    """Sync complet en fin de run (inclut les .keras)."""
    drive_run_dir = os.path.join(DRIVE_DIR, run_name)
    shutil.copytree(run_dir, drive_run_dir, dirs_exist_ok=True)

    global_csv = os.path.join(RUNS_DIR, 'all_runs_summary.csv')
    if os.path.exists(global_csv):
        shutil.copy(global_csv, os.path.join(DRIVE_DIR, 'all_runs_summary.csv'))
    print(f"  [DRIVE] Sync final → {drive_run_dir}")


def update_global_summary(config, run_name, timestamp,
                           total_params, best_loss, best_policy_acc, epochs_run):
    """Ajoute une ligne au CSV global all_runs_summary.csv."""
    global_csv    = os.path.join(RUNS_DIR, 'all_runs_summary.csv')
    global_fields = ['run_name', 'model_id', 'model_name', 'params',
                     'best_loss', 'best_policy_acc',
                     'lr', 'seed', 'epochs_run', 'timestamp']
    file_exists = os.path.exists(global_csv)

    with open(global_csv, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=global_fields)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'run_name'        : run_name,
            'model_id'        : config['model_id'],
            'model_name'      : config['model_name'],
            'params'          : total_params,
            'best_loss'       : round(best_loss, 6),
            'best_policy_acc' : round(best_policy_acc, 6),
            'lr'              : config['lr'],
            'seed'            : config['seed'],
            'epochs_run'      : epochs_run,
            'timestamp'       : timestamp,
        })


def print_final_summary(config, run_dir, run_name,
                         best_loss, best_policy_acc, total_params, epochs_run):
    print(f"\n{'='*60}")
    print(f"  RUN TERMINÉ : {run_name}")
    print(f"  Drive       : {DRIVE_DIR}/{run_name}")
    print(f"  Best loss        : {best_loss:.4f}")
    print(f"  Best policy acc  : {best_policy_acc:.4f}")
    print(f"  Epochs réelles   : {epochs_run}/{config['epochs']}")
    print(f"  Params           : {total_params:,}")
    print(f"  Outputs          : {run_dir}/")
    print(f"    ├── config.txt")
    print(f"    ├── history.csv          (train, toutes les epochs)")
    print(f"    ├── val_results.csv      (val fixe, tous les 10 epochs)")
    print(f"    ├── best_model.keras")
    print(f"    ├── logs/                (TensorBoard)")
    print(f"    ├── curves_epoch[n].png  (plots intermédiaires)")
    print(f"    └── report_visuals.png   (graphiques rapport)")
    print(f"  Global : {RUNS_DIR}/all_runs_summary.csv")
    print(f"{'='*60}\n")

# ───────────────────────────────────────────────────────────────────
# FONCTION D'ENTRAÎNEMENT UNIVERSELLE
# ───────────────────────────────────────────────────────────────────
def train_model(build_fn, config):
    """
    Entraîne n'importe quel modèle compilé avec le socle standard.
    build_fn(config) est appelé dans strategy.scope().
    """
    set_seeds(config['seed'])
    with strategy.scope():
        model = build_fn(config)

    # ── Setup run ────────────────────────────────────────────────────
    run_dir, run_name, timestamp = make_run_dir(config)
    wandb.init(
        project = "go-19x19",
        name    = run_name,
        config  = config,
        reinit  = True
    )
    csv_path, csv_fields, val_path, val_fields = make_csv(run_dir)
    history_log = {k: [] for k in csv_fields}

    checkpoint_path = os.path.join(run_dir, 'best_model.keras')
    log_dir         = os.path.join(run_dir, 'logs')

    # ── Callbacks ────────────────────────────────────────────────────
    callbacks = [
        keras.callbacks.ReduceLROnPlateau(
            monitor='loss', factor=0.5,
            patience=config['patience'], mode='min',
            min_lr=1e-6, verbose=1
        ),
    ]
    tb_writer = tf.summary.create_file_writer(log_dir)

    # ── Boucle ───────────────────────────────────────────────────────
    best_loss          = float('inf')
    best_policy_acc    = 0.0
    early_stop_counter = 0

    for i in range(1, config['epochs'] + 1):
        print(f"\n{'─'*50}")
        print(f"  Epoch {i}/{config['epochs']}")
        print(f"{'─'*50}")

        golois.getBatch(input_data, policy, value, end, groups, i * N)

        hist = model.fit(
            input_data, [policy, value],
            initial_epoch=i - 1,
            epochs=i,
            batch_size=config['batch'],
            callbacks=callbacks,
            verbose=1
        )

        # ── Métriques ────────────────────────────────────────────────
        h = hist.history
        row = {
            'epoch'                       : i,
            'loss'                        : h['loss'][0],
            'policy_loss'                 : h['policy_loss'][0],
            'value_loss'                  : h['value_loss'][0],
            'policy_categorical_accuracy' : h['policy_categorical_accuracy'][0],
            'value_mae'                   : h['value_mae'][0],
        }

        with open(csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=csv_fields).writerow(row)
        for k, v in row.items():
            history_log[k].append(v)

        if row['policy_categorical_accuracy'] > best_policy_acc:
            best_policy_acc = row['policy_categorical_accuracy']

        print(f"  loss={row['loss']:.4f} | "
              f"policy_acc={row['policy_categorical_accuracy']:.4f} | "
              f"value_mae={row['value_mae']:.4f}")

        # ── TensorBoard ──────────────────────────────────────────────
        with tb_writer.as_default():
            tf.summary.scalar('loss/total',  row['loss'],                        step=i)
            tf.summary.scalar('loss/policy', row['policy_loss'],                 step=i)
            tf.summary.scalar('loss/value',  row['value_loss'],                  step=i)
            tf.summary.scalar('acc/policy',  row['policy_categorical_accuracy'], step=i)
            tf.summary.scalar('mae/value',   row['value_mae'],                   step=i)
        tb_writer.flush()

        # ── Val loss (chaque epoch — pour checkpoint et early stopping) ─
        val = model.evaluate(
            val_input, [val_policy, val_value],
            verbose=0, batch_size=config['batch']
        )

        current_val_loss       = val[0]   
        current_val_policy_acc = val[3]   
        
        wandb.log({
            'epoch'            : i,
            'train_loss'       : row['loss'],
            'train_policy_acc' : row['policy_categorical_accuracy'],
            'train_value_mae'  : row['value_mae'],
            'val_loss'         : current_val_loss,        
            'val_policy_acc'   : current_val_policy_acc,  
            'lr'               : float(keras.backend.get_value(
                                     model.optimizer.learning_rate))
        })

        # ── Early stopping ───────────────────────────────────────────────
        if current_val_loss < best_loss:
            best_loss          = current_val_loss
            early_stop_counter = 0
            model.save(checkpoint_path)  # ← save manuel basé sur val_loss
            print(f"  [CHECKPOINT] val_loss → {best_loss:.4f} — modèle sauvegardé")
        else:
            early_stop_counter += 1
            print(f"  [EarlyStopping] patience {early_stop_counter}/{config['patience']}")
            if early_stop_counter >= config['patience']:
                print(f"  [EarlyStopping] Arrêt à l'époque {i}.")
                break

        if i % 5 == 0:
            gc.collect()

        # ── Val + plot + sync tous les 10 epochs ─────────────────────
        if i % 10 == 0:
            print(f"\n  [VAL epoch {i}] total={current_val_loss:.4f} | "
                  f"policy_acc={val[3]:.4f} | value_mae={val[4]:.4f}")

            with open(val_path, 'a', newline='') as f:
                csv.DictWriter(f, fieldnames=val_fields).writerow({
                    'epoch'                       : i,
                    'total_loss'                  : current_val_loss,
                    'policy_loss'                 : val[1],
                    'value_loss'                  : val[2],
                    'policy_categorical_accuracy' : val[3],
                    'value_mae'                   : val[4],
                })

            plot_curves(history_log, run_dir, run_name, i)
            drive_sync_light(run_dir, run_name)

    tb_writer.close()

    # ── Fin de run ───────────────────────────────────────────────────
    epochs_run = len(history_log['epoch'])
    save_final_plots(history_log, run_dir)
    update_global_summary(config, run_name, timestamp,
                          model.count_params(), best_loss, best_policy_acc, epochs_run)
    drive_sync_full(run_dir, run_name)
    print_final_summary(config, run_dir, run_name,
                        best_loss, best_policy_acc, model.count_params(), epochs_run)
    wandb.finish()
    return history_log, best_loss, best_policy_acc


print("✅ Cellule BASE chargée — prêt pour les modèles")

2026-03-29 15:38:27.572145: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774798707.767279      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774798707.820982      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774798708.231726      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774798708.231771      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774798708.231774      24 computation_placer.cc:177] computation placer alr

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs détectés : 2
TensorFlow : 2.19.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Initialisation des données...


I0000 00:00:1774798732.465630      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774798732.471668      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


✅ Val set fixe chargé — ne sera plus modifié

✅ Cellule BASE chargée — prêt pour les modèles


r.shape = (10000, 19, 19, 31)
nbExamples = 10000
nbPositionsSGF = 29425326
nbPositionsSGF = 29425326
generating validation.data


# M1 : MLP Baseline

In [5]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M1 — MLP Baseline | Projet Go 19×19
#  Flatten → Dense(8) → Dense(16) → Dense(26) → dual head
#  ≈ 99 904 params
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 1,
    'model_name' : 'mlp_baseline',
    'planes'     : PLANES,      # défini dans BASE
    'moves'      : MOVES,       # défini dans BASE
    'N'          : N,           # défini dans BASE
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'hidden1'    : 8,
    'hidden2'    : 16,
    'hidden3'    : 26,
}

def build_model(config):
    reg = regularizers.l2(config['l2'])

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')
    x   = layers.Flatten()(inp)
    x   = layers.Dense(config['hidden1'], activation='relu', kernel_regularizer=reg)(x)
    x   = layers.Dense(config['hidden2'], activation='relu', kernel_regularizer=reg)(x)
    x   = layers.Dense(config['hidden3'], activation='relu', kernel_regularizer=reg)(x)

    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(x)
    value_head  = layers.Dense(1, activation='sigmoid',
                                name='value',  kernel_regularizer=reg,
                               dtype='float32')(x)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m1, best_loss_m1, best_acc_m1 = train_model(build_model, config)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 11191)     │          0 │ board[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 8)         │     89,536 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        144 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 26)        │        442 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ policy (Dense)      │ (None, 361)       │      9,747 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         27 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 99,896 (390.22 KB)

 Trainable params: 99,896 (390.22 KB)

 Non-trainable params: 0 (0.00 B)

...

2026-03-29 15:59:28.836408: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 5.8903 - policy_categorical_accuracy: 0.0049 - policy_loss: 5.7631 - value_loss: 0.1150 - value_mae: 0.2809 - learning_rate: 0.0010
  loss=5.8972 | policy_acc=0.0049 | value_mae=0.2843
  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 143.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model1_mlp_baseline_lr0.001_seed42_20260329_153917

  RUN TERMINÉ : model1_mlp_baseline_lr0.001_seed42_20260329_153917
  Drive       : /kaggle/working/go_project3/model1_mlp_baseline_lr0.001_seed42_20260329_153917
  Best loss        : 5.8944
  Best policy acc  : 0.0067
  Epochs réelles   : 143/300
  Params           : 99,896
  Outputs          : runs/model1_mlp_baseline_lr0.001_seed42_20260329_153917/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▇▆▆▆▅▄▄▃▃▃▂▃▃▂▃▁▃▂▂▁▂▂▁▁▂▂▁▃▂▂▂▂▂▁▂▂▁▁▂
wandb: train_policy_acc ▁▁▄▄▄▂▄▅▅▄▅▄▄▃▅▆▄▆█▇▃▃▄▄▆█▆▃▃▃▃▄▃▄▄▇▅▆▆▄
wandb:  train_value_mae █▅▅▅▅▂▆▄▅▄▆▅▃▄▄▆▇▄▄▅▂▄▄▃▁▄▃▄▅▄▄▄▂▅▄▆▅▄▂▄
wandb:         val_loss █▇▇▆▄▃▃▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▂▁▃▅▄▇▄▄▆▁▆▅▆▆▇▅▅▄▅▆▇▅▆▆▆▄▇▆█▄▅▄▄▆█▇▄▅▅▅
wandb: 
wandb: Run summary:
wandb:            epoch 143
wandb:               lr 0.001
wandb:       train_loss 5.89718
wandb: train_policy_acc 0.0049
wandb:  train_value_mae 0.28428
wandb:         val_loss 5.89879
wandb:   val_policy_acc 0.005
wandb: 
wandb: 🚀 View run model1_mlp_baseline_lr0.001_seed42_20260329_153917 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/s0w94tw3
wandb: ⭐️ View project at: https://wandb.ai/rouibiamine7-universit-paris-da

# M2 : CNN Shallow

In [6]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M2 — CNN Shallow | Projet Go 19×19
#  Conv(32)×3 [3×3, same] → policy: Conv(1)+Dense(64)+out
#                          → value:  Conv(1)+Dense(64)+out
#  ≈ 97 388 params
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 2,
    'model_name' : 'cnn_shallow',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 32,
    'dense_head' : 64,
}

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Backbone ─────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(inp)
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(x)
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(x)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m2, best_loss_m2, best_acc_m2 = train_model(build_model, config)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 19, 19,    │      8,960 │ board[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 19, 19,    │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 19, 19,    │      9,248 │ conv2d_1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 19, 19, 1) │         33 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 19, 19, 1) │         33 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 361)       │          0 │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 361)       │          0 │ conv2d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │     23,168 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │     23,168 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ policy (Dense)      │ (None, 361)       │     23,465 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         65 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 97,388 (380.42 KB)

 Trainable params: 97,388 (380.42 KB)

 Non-trainable params: 0 (0.00 B)

...

2026-03-29 16:31:03.180602: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  loss=4.5049 | policy_acc=0.1022 | value_mae=0.2819
  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 215.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model2_cnn_shallow_lr0.001_seed42_20260329_155936

  RUN TERMINÉ : model2_cnn_shallow_lr0.001_seed42_20260329_155936
  Drive       : /kaggle/working/go_project3/model2_cnn_shallow_lr0.001_seed42_20260329_155936
  Best loss        : 4.4758
  Best policy acc  : 0.1101
  Epochs réelles   : 215/300
  Params           : 97,388
  Outputs          : runs/model2_cnn_shallow_lr0.001_seed42_20260329_155936/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading history steps 213-214, summary, console lines 2544-2579
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▇▇▇▇▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train_policy_acc ▁▁▁▁▁▁▁▂▃▃▄▄▅▄▅▅▆▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇█████
wandb:  train_value_mae ▆█▅▆▆▄▄▅▆▇▃▄▅▅▇▂▂▄▆▅▃▆▆▄▅▃▄▃▂▆▄▁▃▄▃▆▅▁▇▂
wandb:         val_loss ███▇▇▇▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▁▁▁▁▁▂▃▃▄▄▅▄▄▅▅▅▆▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████
wandb: 
wandb: Run summary:
wandb:            epoch 215
wandb:               lr 0.001
wandb:       train_loss 4.50495
wandb: train_policy_acc 0.1022
wandb:  train_value_mae 0.28192
wandb:         val_loss 4.50815
wandb:   val_policy_acc 0.1086
wandb: 
wandb: 🚀 View run model2_cnn_shallow_lr0.001_seed42_20260329_155936 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/qbakkzbb
wandb: ⭐️ View project at: https://wan

# M3 : CNN + BatchNorm

In [7]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M3 — CNN + BatchNorm | Projet Go 19×19
#  Identique M2 + BN après chaque Conv
#  dense_head : 64→66 pour maximiser sous 100k
#  ≈ 99 752 params
#
#  Budget :
#    M2 base         = 97 388
#    BN ×3 (64/layer)=    192  → 97 580
#    dense_head 64→66=  2 172  → 99 752  ✓ < 100k
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 3,
    'model_name' : 'cnn_batchnorm',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 32,
    'dense_head' : 66,          # ← 64 en M2, monté à 66 pour maximiser
}

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Backbone : Conv + BN après chaque couche ─────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m3, best_loss_m3, best_acc_m3 = train_model(build_model, config)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 19, 19,    │      8,928 │ board[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 19, 19,    │        128 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 19, 19,    │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │        128 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 19, 19,    │      9,216 │ activation_1[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │        128 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 19, 19, 1) │         33 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 19, 19, 1) │         33 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 361)       │          0 │ conv2d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 361)       │          0 │ conv2d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 66)        │     23,892 │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 66)        │     23,892 │ flatten_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ policy (Dense)      │ (None, 361)       │     24,187 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         67 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 99,848 (390.03 KB)

 Trainable params: 99,656 (389.28 KB)

 Non-trainable params: 192 (768.00 B)

...

2026-03-29 16:50:46.453933: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  loss=nan | policy_acc=0.0014 | value_mae=nan
  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 131.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model3_cnn_batchnorm_lr0.001_seed42_20260329_163109

  RUN TERMINÉ : model3_cnn_batchnorm_lr0.001_seed42_20260329_163109
  Drive       : /kaggle/working/go_project3/model3_cnn_batchnorm_lr0.001_seed42_20260329_163109
  Best loss        : 3.8736
  Best policy acc  : 0.2943
  Epochs réelles   : 131/300
  Params           : 99,848
  Outputs          : runs/model3_cnn_batchnorm_lr0.001_seed42_20260329_163109/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading history steps 130-130, summary, console lines 1547-1579
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss ███████▅▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁     
wandb: train_policy_acc ▁▁▁▁▁▁▁▁▁▁▂▂▃▃▃▄▄▅▅▅▇▇▇▇█████████████▁▁▁
wandb:  train_value_mae █▄▄▄▂▁▃▂▃▃▄▃▃▂▄▄▂▂▅▃▃▂▃▃▂▂▃▄▃▃▃▂▂▄▄     
wandb:         val_loss █████████▇▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁         
wandb:   val_policy_acc ▁▁▁▁▁▁▁▁▁▁▂▂▂▃▂▃▃▃▄▄▅▆▇▇▇▇█████████▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 131
wandb:               lr 0.001
wandb:       train_loss nan
wandb: train_policy_acc 0.0014
wandb:  train_value_mae nan
wandb:         val_loss nan
wandb:   val_policy_acc 0.0017
wandb: 
wandb: 🚀 View run model3_cnn_batchnorm_lr0.001_seed42_20260329_163109 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/k5jqv988
wandb: ⭐️ View project at: https://wandb.ai/roui

# M4 : ResNet Tiny

In [8]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M4 — ResNet Tiny | Projet Go 19×19
#  Conv(24) → [ResBlock(24)]×3 → dual head(fc=56)
#
#  Budget :
#    Initial Conv(24,3×3)           =  6 720
#    ResBlock×3 (BN, no skip params)= 31 680
#    Policy Conv(1)+Dense(56)+out   = 40 874
#    Value  Conv(1)+Dense(56)+out   = 20 354
#    ─────────────────────────────────────────
#    Total                          = 99 628  ✓ < 100k
#    (fc=48 décrit → 90 940 | fc=57 → 100 786 ❌)
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 4,
    'model_name' : 'resnet_tiny',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 24,
    'n_blocks'   : 3,
    'dense_head' : 56,          # ← 48 en description, monté à 56 pour maximiser
}

def resblock(x, filters, reg):
    """Conv(no_bias)+BN+ReLU+Conv(no_bias)+BN + skip identity."""
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization(gamma_initializer='zeros')(x)
    x = layers.Add()([x, shortcut])   # skip connection — 0 params
    x = layers.Activation('relu')(x)
    return x

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Stem ─────────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # ── ResBlocks ────────────────────────────────────────────────────
    for _ in range(config['n_blocks']):
        x = resblock(x, f, reg)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m4, best_loss_m4, best_acc_m4 = train_model(build_model, config)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 19, 19,    │      6,696 │ board[0][0]       │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_3[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_11[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_4[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 19, 19,    │          0 │ batch_normalizat… │
│                     │ 24)               │            │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 19, 19,    │          0 │ add[0][0]         │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_5[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_13[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_6[0][… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_14[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 19, 19,    │          0 │ batch_normalizat

 Total params: 99,700 (389.45 KB)

 Trainable params: 99,364 (388.14 KB)

 Non-trainable params: 336 (1.31 KB)

...

2026-03-29 17:23:35.534240: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  loss=nan | policy_acc=0.0014 | value_mae=nan
  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 208.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model4_resnet_tiny_lr0.001_seed42_20260329_165053

  RUN TERMINÉ : model4_resnet_tiny_lr0.001_seed42_20260329_165053
  Drive       : /kaggle/working/go_project3/model4_resnet_tiny_lr0.001_seed42_20260329_165053
  Best loss        : 3.8018
  Best policy acc  : 0.3145
  Epochs réelles   : 208/300
  Params           : 99,700
  Outputs          : runs/model4_resnet_tiny_lr0.001_seed42_20260329_165053/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss ██████▇▆▅▄▄▄▄▄▃▃▃▂▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁    
wandb: train_policy_acc ▁▁▁▁▁▁▁▁▁▁▂▃▃▃▃▃▃▃▃▄▅▅▆▆▆▇▇▇▇▇▇███▇████▁
wandb:  train_value_mae █▆▃▃▅▆▃▅▅▇▅▆▆▇▃▆▅▇▅▅▇▄▇▆▁▂▆▄▅▄▅▄▇▅▆▃▆   
wandb:         val_loss ████████▇▆▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁   
wandb:   val_policy_acc ▁▁▁▁▁▁▁▁▁▁▁▂▂▃▄▅▅▆▆▆▆▇▇▇▇▇████████▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 208
wandb:               lr 0.001
wandb:       train_loss nan
wandb: train_policy_acc 0.0014
wandb:  train_value_mae nan
wandb:         val_loss nan
wandb:   val_policy_acc 0.0017
wandb: 
wandb: 🚀 View run model4_resnet_tiny_lr0.001_seed42_20260329_165053 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/8h0o4ydu
wandb: ⭐️ View project at: https://wandb.a

# M5 : Depthwise Separable CNN

In [9]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M5 — Depthwise Separable CNN | Projet Go 19×19
#  DSConv(32)×5 [depthwise 3×3 + pointwise 1×1] → dual head(fc=85)
#
#  Budget :
#    DSConv layer1 (31→32) dw+pw   =  1 334
#    DSConv layers2-5 (32→32) ×4   =  5 504
#    Backbone total                 =  6 838
#    Heads (428 + 1086×85)          = 92 738
#    ─────────────────────────────────────────
#    Total                          = 99 576  ✓ < 100k
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 5,
    'model_name' : 'dsconv',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 32,
    'n_layers'   : 5,
    'dense_head' : 85,
}

def ds_block(x, filters, reg):
    """Depthwise 3×3 + Pointwise 1×1 + ReLU."""
    x = layers.DepthwiseConv2D((3, 3), padding='same',
                                depthwise_regularizer=reg)(x)
    x = layers.Conv2D(filters, (1, 1), padding='same',
                      kernel_regularizer=reg)(x)
    x = layers.Activation('relu')(x)
    return x

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Backbone : 5 DSConv blocks ───────────────────────────────────
    x = ds_block(inp, f, reg)               # layer 1 : 31→32
    for _ in range(config['n_layers'] - 1): # layers 2-5 : 32→32
        x = ds_block(x, f, reg)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m5, best_loss_m5, best_acc_m5 = train_model(build_model, config)

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 19, 19,    │        310 │ board[0][0]       │
│ (DepthwiseConv2D)   │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 19, 19,    │      1,024 │ depthwise_conv2d… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 19, 19,    │          0 │ conv2d_19[0][0]   │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_1  │ (None, 19, 19,    │        320 │ activation_10[0]… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 19, 19,    │      1,056 │ depthwise_conv2d… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_11       │ (None, 19, 19,    │          0 │ conv2d_20[0][0]   │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_2  │ (None, 19, 19,    │        320 │ activation_11[0]… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 19, 19,    │      1,056 │ depthwise_conv2d… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_12       │ (None, 19, 19,    │          0 │ conv2d_21[0][0]   │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_3  │ (None, 19, 19,    │        320 │ activation_12[0]… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 19, 19,    │      1,056 │ depthwise_conv2d… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_13       │ (None, 19, 19,    │          0 │ conv2d_22[0][0]   │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_4  │ (None, 19, 19,    │        320 │ activation_13[0]… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 19, 19,    │      1,056 │ depthwise_conv2d… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_14       │ (None, 19, 19,    │          0 │ conv2d_23[0][0]   │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 19, 19, 1) │         33 │ activation_14[0]

 Total params: 99,576 (388.97 KB)

 Trainable params: 99,576 (388.97 KB)

 Non-trainable params: 0 (0.00 B)

...

2026-03-29 17:30:06.492202: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  loss=6.0003 | policy_acc=0.0038 | value_mae=0.2900
  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 41.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model5_dsconv_lr0.001_seed42_20260329_172343

  RUN TERMINÉ : model5_dsconv_lr0.001_seed42_20260329_172343
  Drive       : /kaggle/working/go_project3/model5_dsconv_lr0.001_seed42_20260329_172343
  Best loss        : 6.0007
  Best policy acc  : 0.0043
  Epochs réelles   : 41/300
  Params           : 99,576
  Outputs          : runs/model5_dsconv_lr0.001_seed42_20260329_172343/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading history steps 40-40, summary, console lines 477-508
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▅▃▃▂▂▂▂▂▂▂▂▂▁▂▁▁▂▁▂▁▂▂▂▂▁▂▁▁▁▁▁▂▂▁▁▁▂▁▁
wandb: train_policy_acc ▄▃▇▃▄▇▇▂▅▆█▅▄█▆▆▆▅▄▄▄▄▅▂▅▅▄▇██▄▃▄▅▄▅▁▅▄▇
wandb:  train_value_mae █▄▆▇▅▂▆▅▄▄▇▆▄▂▆▃▂█▁▆▄▆▅▆▄▂▇▃▃▄▃▃▃▅▂▆▆▇▄▃
wandb:         val_loss █▅▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▁▃▃▄▅▅▅▂▅▅▅▅▅▅▁▁▁▁▁▆▆▁▆▁▅▅█▃▃▃▃▃▆▆▃▆▅▅▅▅
wandb: 
wandb: Run summary:
wandb:            epoch 41
wandb:               lr 0.001
wandb:       train_loss 6.00032
wandb: train_policy_acc 0.0038
wandb:  train_value_mae 0.29001
wandb:         val_loss 6.00134
wandb:   val_policy_acc 0.0033
wandb: 
wandb: 🚀 View run model5_dsconv_lr0.001_seed42_20260329_172343 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/2zpzvvk5
wandb: ⭐️ View project at: https://wandb.ai/roui

# M6 : CNN Asymétrique (policy head lourde)

In [10]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M6 — CNN Asymétrique (policy head lourde) | Projet Go 19×19
#  Conv(24)×3 → policy: Conv(1)+Dense(87)+Dense(87)+out
#             → value:  Conv(1)+Dense(32)+out
#
#  Budget :
#    Backbone Conv(24)×3            = 17 136
#    Value Conv(1)+Dense(32)+out    = 11 642
#    Policy Conv(1)+Dense(87)×2+out = 70 943
#    ─────────────────────────────────────────
#    Total                          = 99 721  ✓ < 100k
#
#  ⚠️  Description originale : Dense(128)+Dense(128) → impossible
#      sous 100k avec ce backbone (policy seule > 109k).
#      Dense(87)+Dense(87) est le maximum réalisable.
#
#  Hypothèse : la policy est plus dure à apprendre → budget
#  concentré sur sa tête (87+87) vs value (32 seulement).
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'        : 6,
    'model_name'      : 'cnn_asymmetric',
    'planes'          : PLANES,
    'moves'           : MOVES,
    'N'               : N,
    'epochs'          : 300,
    'batch'           : 128 * NUM_GPUS,
    'lr'              : 0.001,
    'l2'              : 0.0001,
    'seed'            : 42,
    'max_params'      : 100_000,
    'patience'        : 20,
    'filters'         : 24,
    'dense_policy'    : 87,     # ← 128 impossible sous 100k, max = 87
    'dense_value'     : 32,
}

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dp  = config['dense_policy']
    dv  = config['dense_value']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Backbone ─────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(inp)
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(x)
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(x)

    # ── Policy head — lourde (2 couches denses) ──────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dp, activation='relu', kernel_regularizer=reg)(p)
    p = layers.Dense(dp, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head — légère (1 couche dense) ─────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dv, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m6, best_loss_m6, best_acc_m6 = train_model(build_model, config)

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 19, 19,    │      6,720 │ board[0][0]       │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 19, 19,    │      5,208 │ conv2d_26[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 19, 19,    │      5,208 │ conv2d_27[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_29 (Conv2D)  │ (None, 19, 19, 1) │         25 │ conv2d_28[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_9 (Flatten) │ (None, 361)       │          0 │ conv2d_29[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, 19, 19, 1) │         25 │ conv2d_28[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 87)        │     31,494 │ flatten_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_10          │ (None, 361)       │          0 │ conv2d_30[0][0]   │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 87)        │      7,656 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 32)        │     11,584 │ flatten_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ policy (Dense)      │ (None, 361)       │     31,768 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         33 │ dense_13[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 99,721 (389.54 KB)

 Trainable params: 99,721 (389.54 KB)

 Non-trainable params: 0 (0.00 B)

...

2026-03-29 17:47:51.491718: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  loss=nan | policy_acc=0.0017 | value_mae=nan
  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 119.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model6_cnn_asymmetric_lr0.001_seed42_20260329_173014

  RUN TERMINÉ : model6_cnn_asymmetric_lr0.001_seed42_20260329_173014
  Drive       : /kaggle/working/go_project3/model6_cnn_asymmetric_lr0.001_seed42_20260329_173014
  Best loss        : 4.8153
  Best policy acc  : 0.0676
  Epochs réelles   : 119/300
  Params           : 99,721
  Outputs          : runs/model6_cnn_asymmetric_lr0.001_seed42_20260329_173014/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading config.yaml
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇████
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss ███▇▇▇▇▇▇▇▇▇▆▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁      
wandb: train_policy_acc ▁▁▁▁▂▁▁▂▂▁▃▃▅▅▅▅▅▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇█▁▁▁▁▁▁▁
wandb:  train_value_mae █▅▅▆▁▁▄▃▃▃▂▄▃▁▂▁▂▂▆▄▄▁▄▂▃▂▄▂▄▃▃▄▃▄▄     
wandb:         val_loss ███▇▇▇▇▇▇▅▄▃▃▃▃▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁        
wandb:   val_policy_acc ▁▁▁▁▁▁▁▁▁▂▁▁▁▂▄▄▅▅▅▅▆▆▆▆▆▇▆▆▇▇▇▇▇▇▇█▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 119
wandb:               lr 0.001
wandb:       train_loss nan
wandb: train_policy_acc 0.0017
wandb:  train_value_mae nan
wandb:         val_loss nan
wandb:   val_policy_acc 0.0017
wandb: 
wandb: 🚀 View run model6_cnn_asymmetric_lr0.001_seed42_20260329_173014 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/hcp5996l
wandb: ⭐️ View project at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19

# M7 : MobileNet-Style (Inverted Residuals)

In [11]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M7 — MobileNet-style (Inverted Residuals) | Projet Go 19×19
#  Conv(16) → InvResBlock(expand×6, 16 filtres)×4 → dual head(fc=72)
#
#  Budget :
#    Stem Conv(16, 3×3)              =  4 480
#    InvResBlock×4 :
#      Expand pw (16→96)  = 1 632   |
#      Depthwise 3×3      =   960   | × 4 = 16 576
#      Project pw (96→16) = 1 552   |
#    Heads (396 + 1086×72)           = 78 588
#    ─────────────────────────────────────────
#    Total                           = 99 644  ✓ < 100k
#    (fc=32 décrit → 56 204 | fc=73 → 100 730 ❌)
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 7,
    'model_name' : 'mobilenet_style',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 16,
    'expand'     : 6,
    'n_blocks'   : 4,
    'dense_head' : 72,          # ← 32 en description, monté à 72 pour maximiser
}

def inv_res_block(x, filters, expand, reg):
    """
    Inverted Residual : Expand → Depthwise → Project + skip identity.
    Skip uniquement si input_channels == output_channels.
    """
    in_channels  = x.shape[-1]
    hidden       = in_channels * expand

    # Expand : pointwise 1×1
    out = layers.Conv2D(hidden, (1, 1), padding='same', use_bias=True,
                        kernel_regularizer=reg)(x)
    out = layers.Activation('relu')(out)

    # Depthwise : 3×3 spatial
    out = layers.DepthwiseConv2D((3, 3), padding='same', use_bias=True,
                                  depthwise_regularizer=reg)(out)
    out = layers.Activation('relu')(out)

    # Project : pointwise 1×1 (pas d'activation — style MobileNetV2)
    out = layers.Conv2D(filters, (1, 1), padding='same', use_bias=True,
                        kernel_regularizer=reg)(out)

    # Skip connection si dimensions identiques
    if in_channels == filters:
        out = layers.Add()([out, x])

    return out

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Stem ─────────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(inp)

    # ── Inverted Residual Blocks ──────────────────────────────────────
    for _ in range(config['n_blocks']):
        x = inv_res_block(x, f, config['expand'], reg)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m7, best_loss_m7, best_acc_m7 = train_model(build_model, config)

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 19, 19,    │      4,480 │ board[0][0]       │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 19, 19,    │      1,632 │ conv2d_31[0][0]   │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_15       │ (None, 19, 19,    │          0 │ conv2d_32[0][0]   │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_5  │ (None, 19, 19,    │        960 │ activation_15[0]… │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_16       │ (None, 19, 19,    │          0 │ depthwise_conv2d… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 19, 19,    │      1,552 │ activation_16[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 19, 19,    │          0 │ conv2d_33[0][0],  │
│                     │ 16)               │            │ conv2d_31[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 19, 19,    │      1,632 │ add_3[0][0]       │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_17       │ (None, 19, 19,    │          0 │ conv2d_34[0][0]   │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_6  │ (None, 19, 19,    │        960 │ activation_17[0]… │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_18       │ (None, 19, 19,    │          0 │ depthwise_conv2d… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, 19, 19,    │      1,552 │ activation_18[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 19, 19,    │          0 │ conv2d_35[0][0],  │
│                     │ 16)               │            │ add_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_36 (Conv2D)  │ (None, 19, 19,    │      1,632 │ add_4[0][0]       │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_19       │ (None, 19, 19,    │          0 │ conv2d_36[0][0]   │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_7  │ (None, 19, 19,    │        960 │ activation_19[0]

 Total params: 99,644 (389.23 KB)

 Trainable params: 99,644 (389.23 KB)

 Non-trainable params: 0 (0.00 B)

...

2026-03-29 18:00:10.225475: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 75.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model7_mobilenet_style_lr0.001_seed42_20260329_174800

  RUN TERMINÉ : model7_mobilenet_style_lr0.001_seed42_20260329_174800
  Drive       : /kaggle/working/go_project3/model7_mobilenet_style_lr0.001_seed42_20260329_174800
  Best loss        : 4.5107
  Best policy acc  : 0.1738
  Epochs réelles   : 75/300
  Params           : 99,644
  Outputs          : runs/model7_mobilenet_style_lr0.001_seed42_20260329_174800/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading history steps 74-74, summary, console lines 879-912
wandb: 
wandb: Run history:
wandb:            epoch ▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss ████████▇▇▇▆▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁          
wandb: train_policy_acc ▁▁▁▁▁▁▁▁▁▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▆▆▆▇█▁▁▁▁▁▁▁▁▁▁
wandb:  train_value_mae █▅▆▂▃▅▄▂▂▃▂▄▂▄▃▃▁▄▃▃▂▃▂▄▂▄▃▁▂▃▂▂▄       
wandb:         val_loss ████████▇▇▇▇▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁       
wandb:   val_policy_acc ▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▆▇█▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 75
wandb:               lr 0.001
wandb:       train_loss nan
wandb: train_policy_acc 0.0016
wandb:  train_value_mae nan
wandb:         val_loss nan
wandb:   val_policy_acc 0.0017
wandb: 
wandb: 🚀 View run model7_mobilenet_style_lr0.001_seed42_20260329_174800 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/nm23u4y5
wandb: ⭐️ View project at: https://wandb.ai/rouibia

# M8 — ResNet + Squeeze-and-Excitation

In [12]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M8 — ResNet + Squeeze-and-Excitation | Projet Go 19×19
#  Conv(24) → [ResBlock(24) + SE(ratio=4)]×3 → dual head(fc=55)
#
#  Budget :
#    Stem Conv(24, 3×3)              =  6 720
#    ResBlock(24) no_bias + BN       = 10 560
#    SE(ratio=4→6) Dense(6)+Dense(24)=    318
#    [ResBlock+SE]×3                 = 32 634
#    Heads (412 + 1086×55)           = 60 142
#    ─────────────────────────────────────────
#    Total                           = 99 496  ✓ < 100k
#    (fc=48 décrit → 92 530 | fc=56 → 100 582 ❌)
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 8,
    'model_name' : 'resnet_se',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 24,
    'n_blocks'   : 3,
    'se_ratio'   : 4,
    'dense_head' : 55,          # ← 48 en description, monté à 55 pour maximiser
}

def se_block(x, filters, ratio, reg):
    """Squeeze-and-Excitation : GAP → Dense(C/ratio) → Dense(C) → scale."""
    se_units = max(1, filters // ratio)
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(se_units, activation='relu',
                      kernel_regularizer=reg)(se)
    se = layers.Dense(filters, activation='sigmoid',
                      kernel_regularizer=reg)(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def resblock_se(x, filters, ratio, reg):
    """Conv+BN+ReLU+Conv+BN → SE → Add skip → ReLU."""
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x, filters, ratio, reg)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Stem ─────────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(inp)

    # ── ResBlock + SE × n_blocks ─────────────────────────────────────
    for _ in range(config['n_blocks']):
        x = resblock_se(x, f, config['se_ratio'], reg)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m8, best_loss_m8, best_acc_m8 = train_model(build_model, config)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_42 (Conv2D)  │ (None, 19, 19,    │      6,720 │ board[0][0]       │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_43 (Conv2D)  │ (None, 19, 19,    │      5,184 │ conv2d_42[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_43[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_23       │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_44 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_23[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_44[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 24)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 6)         │        150 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 24)        │        168 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 24)  │          0 │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 19, 19,    │          0 │ batch_normalizat… │
│                     │ 24)               │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 19, 19,    │          0 │ multiply[0][0],   │
│                     │ 24)               │            │ conv2d_42[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_24       │ (None, 19, 19,    │          0 │ add_7[0][0]       │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_45 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_24[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_45[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_25       │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_46 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_25[0]

 Total params: 99,496 (388.66 KB)

 Trainable params: 99,208 (387.53 KB)

 Non-trainable params: 288 (1.12 KB)

...

2026-03-29 18:12:59.652749: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 77.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model8_resnet_se_lr0.001_seed42_20260329_180015

  RUN TERMINÉ : model8_resnet_se_lr0.001_seed42_20260329_180015
  Drive       : /kaggle/working/go_project3/model8_resnet_se_lr0.001_seed42_20260329_180015
  Best loss        : 4.3627
  Best policy acc  : 0.1751
  Epochs réelles   : 77/300
  Params           : 99,496
  Outputs          : runs/model8_resnet_se_lr0.001_seed42_20260329_180015/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇██
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss ███████████▇█▇▇▆▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁         
wandb: train_policy_acc ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▄▄▄▅▅▆▆▆▇█▁▁▁▁▁▁▁▁▁▁
wandb:  train_value_mae █▅▄▂▃▂▃▂▁▂▄▃▃▃▅▃▄▂▃▂▃▃▁▅▄▄▂▂▃▅          
wandb:         val_loss ████████████▇▇▇▆▄▄▄▃▃▃▃▃▃▃▂▂▂▁▁         
wandb:   val_policy_acc ▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▄▄▄▄▄▄▅▅▅▅▇▇▇██▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 77
wandb:               lr 0.001
wandb:       train_loss nan
wandb: train_policy_acc 0.0012
wandb:  train_value_mae nan
wandb:         val_loss nan
wandb:   val_policy_acc 0.0017
wandb: 
wandb: 🚀 View run model8_resnet_se_lr0.001_seed42_20260329_180015 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/bscuyo3r
wandb: ⭐️ View project at: https://wandb.ai/r

# M9 : ResNet + Dilated Convolutions

In [13]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M9 — ResNet + Dilated Convolutions | Projet Go 19×19
#  Conv(24) → ResBlock(d=1) → ResBlock(d=2) → ResBlock(d=3)
#           → dual head(fc=56)
#
#  Budget :
#    Stem Conv(24, 3×3)              =  6 720
#    ResBlock×3 no_bias+BN           = 31 680
#    (dilation ne change pas les params)
#    Heads (412 + 1086×56)           = 61 228 - wait
#    412 + 1086×56 = 412 + 60 816   = 61 228
#    ─────────────────────────────────────────
#    Total 38 400 + 61 228           = 99 628  ✓ < 100k
#    (fc=57 → 100 714 ❌)
#
#  Champ récepteur effectif :
#    d=1 : 3×3  → 3
#    d=2 : 3×3  → 7  (joints avec d=1 : 9)
#    d=3 : 3×3  → 13 (joints avec d=1+2 : 21)
#    Capture les interactions à longue distance sur le goban 19×19.
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 9,
    'model_name' : 'resnet_dilated',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 24,
    'dilations'  : [1, 2, 3],   # un ResBlock par taux de dilatation
    'dense_head' : 56,          # ← maximisé (fc=57 → 100 714 ❌)
}

def resblock_dilated(x, filters, dilation_rate, reg):
    """
    Conv(dilation)+BN+ReLU + Conv(dilation)+BN + skip + ReLU.
    Même nombre de params qu'un ResBlock standard — seul le champ
    récepteur change.
    """
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      dilation_rate=dilation_rate,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False,
                      dilation_rate=dilation_rate,
                      kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    dh  = config['dense_head']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── Stem ─────────────────────────────────────────────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(inp)

    # ── ResBlocks avec dilatations croissantes ────────────────────────
    for d in config['dilations']:
        x = resblock_dilated(x, f, d, reg)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m9, best_loss_m9, best_acc_m9 = train_model(build_model, config)

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_51 (Conv2D)  │ (None, 19, 19,    │      6,720 │ board[0][0]       │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_52 (Conv2D)  │ (None, 19, 19,    │      5,184 │ conv2d_51[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_52[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_29       │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_53 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_29[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_53[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│                     │ 24)               │            │ conv2d_51[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_30       │ (None, 19, 19,    │          0 │ add_10[0][0]      │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_54 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_30[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_54[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_31       │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_55 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_31[0]… │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │         96 │ conv2d_55[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 19, 19,    │          0 │ batch_normalizat… │
│                     │ 24)               │            │ activation_30[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_32       │ (None, 19, 19,    │          0 │ add_11[0][0]      │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_56 (Conv2D)  │ (None, 19, 19,    │      5,184 │ activation_32[0]

 Total params: 99,628 (389.17 KB)

 Trainable params: 99,340 (388.05 KB)

 Non-trainable params: 288 (1.12 KB)

...

2026-03-29 18:55:16.552675: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 263.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model9_resnet_dilated_lr0.001_seed42_20260329_181305

  RUN TERMINÉ : model9_resnet_dilated_lr0.001_seed42_20260329_181305
  Drive       : /kaggle/working/go_project3/model9_resnet_dilated_lr0.001_seed42_20260329_181305
  Best loss        : 3.6661
  Best policy acc  : 0.3270
  Epochs réelles   : 263/300
  Params           : 99,628
  Outputs          : runs/model9_resnet_dilated_lr0.001_seed42_20260329_181305/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇███
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █████▇▇▇▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ 
wandb: train_policy_acc ▁▁▁▁▁▁▁▁▁▂▃▃▄▅▅▇▇▇▇▇▇▇▇█▇██████████████▁
wandb:  train_value_mae █▂▁▂▅▅▂▆▆▇▅▇▅█▅▂▆▄▄▅▅▅▃▄▄▄▃▅▆▆▂▅▁▄▇▂▃▃▂▃
wandb:         val_loss ██████████▅▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁  
wandb:   val_policy_acc ▁▁▁▁▁▁▁▁▂▂▃▃▃▄▅▆▆▆▇▇▇▇███████████████▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 263
wandb:               lr 0.001
wandb:       train_loss nan
wandb: train_policy_acc 0.0011
wandb:  train_value_mae nan
wandb:         val_loss nan
wandb:   val_policy_acc 0.0017
wandb: 
wandb: 🚀 View run model9_resnet_dilated_lr0.001_seed42_20260329_181305 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/3zm1bc6f
wandb: ⭐️ View project at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19
wandb: Synced 5 W&B file(

# M10 : CNN + Transformer Encoder

In [14]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M10 — CNN + Transformer Encoder | Projet Go 19×19
#  Conv(16)×2 → Reshape(361,16) → LayerNorm+MHA(4,16)+skip
#             → Reshape(19,19,16) → dual head(fc=81)
#
#  Budget :
#    CNN Conv(16)×2                  =  6 800
#    LayerNorm(16)                   =     32
#    MHA(heads=4, key_dim=16)        =  4 304
#      W_q/W_k/W_v : 3×(16×64+64)  = 3 264
#      W_o         : 64×16+16       = 1 040
#    Heads (396 + 1086×81)           = 88 362
#    ─────────────────────────────────────────
#    Total                           = 99 498  ✓ < 100k
#    (fc=82 → 100 584 ❌)
#
#  Chaque intersection du goban = 1 token de dim 16.
#  L'attention capte les dépendances globales (atari, ladders...)
#  impossibles à voir avec une conv locale.
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 10,
    'model_name' : 'cnn_transformer',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 16,
    'mha_heads'  : 4,
    'key_dim'    : 16,
    'dense_head' : 81,          # ← 64 en description, monté à 81 pour maximiser
}

def build_model(config):
    reg  = regularizers.l2(config['l2'])
    f    = config['filters']
    dh   = config['dense_head']
    seq  = 19 * 19              # 361 tokens

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    # ── CNN Backbone — extraction de features locales ─────────────────
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(inp)
    x = layers.Conv2D(f, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=reg)(x)

    # ── Reshape : (19,19,16) → (361,16) pour le Transformer ──────────
    x = layers.Reshape((seq, f))(x)

    # ── Transformer Encoder Block (pre-norm style) ────────────────────
    #    LayerNorm → MHA → Add skip (résiduel)
    residual = x
    x = layers.LayerNormalization()(x)
    x = layers.MultiHeadAttention(
            num_heads=config['mha_heads'],
            key_dim=config['key_dim']
        )(x, x)                         # self-attention
    x = layers.Add()([x, residual])     # skip connection

    # ── Reshape : (361,16) → (19,19,16) pour les têtes Conv ──────────
    x = layers.Reshape((19, 19, f))(x)

    # ── Policy head ──────────────────────────────────────────────────
    p = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    p = layers.Flatten()(p)
    p = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(p)
    policy_head = layers.Dense(config['moves'], activation='softmax',
                                name='policy', kernel_regularizer=reg,
                                dtype='float32')(p)

    # ── Value head ───────────────────────────────────────────────────
    v = layers.Conv2D(1, (1, 1), activation='relu', kernel_regularizer=reg)(x)
    v = layers.Flatten()(v)
    v = layers.Dense(dh, activation='relu', kernel_regularizer=reg)(v)
    value_head = layers.Dense(1, activation='sigmoid',
                               name='value', kernel_regularizer=reg,
                               dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model

# ── Lancement ────────────────────────────────────────────────────────
history_m10, best_loss_m10, best_acc_m10 = train_model(build_model, config)

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 19, 19,    │          0 │ -                 │
│                     │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_60 (Conv2D)  │ (None, 19, 19,    │      4,480 │ board[0][0]       │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_61 (Conv2D)  │ (None, 19, 19,    │      2,320 │ conv2d_60[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 361, 16)   │          0 │ conv2d_61[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 361, 16)   │         32 │ reshape_3[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 361, 16)   │      4,304 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (None, 361, 16)   │          0 │ multi_head_atten… │
│                     │                   │            │ reshape_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_4 (Reshape) │ (None, 19, 19,    │          0 │ add_13[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_62 (Conv2D)  │ (None, 19, 19, 1) │         17 │ reshape_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_63 (Conv2D)  │ (None, 19, 19, 1) │         17 │ reshape_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_17          │ (None, 361)       │          0 │ conv2d_62[0][0]   │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_18          │ (None, 361)       │          0 │ conv2d_63[0][0]   │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 81)        │     29,322 │ flatten_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 81)        │     29,322 │ flatten_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ policy (Dense)      │ (None, 361)       │     29,602 │ dense_26[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         82 │ dense_27[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 99,498 (388.66 KB)

 Trainable params: 99,498 (388.66 KB)

 Non-trainable params: 0 (0.00 B)

...

2026-03-29 19:07:26.728709: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


  [EarlyStopping] patience 20/20
  [EarlyStopping] Arrêt à l'époque 62.


wandb: updating run metadata


✅ report_visuals.png sauvegardé
  [DRIVE] Sync final → /kaggle/working/go_project3/model10_cnn_transformer_lr0.001_seed42_20260329_185524

  RUN TERMINÉ : model10_cnn_transformer_lr0.001_seed42_20260329_185524
  Drive       : /kaggle/working/go_project3/model10_cnn_transformer_lr0.001_seed42_20260329_185524
  Best loss        : 6.0002
  Best policy acc  : 0.0046
  Epochs réelles   : 62/300
  Params           : 99,498
  Outputs          : runs/model10_cnn_transformer_lr0.001_seed42_20260329_185524/
    ├── config.txt
    ├── history.csv          (train, toutes les epochs)
    ├── val_results.csv      (val fixe, tous les 10 epochs)
    ├── best_model.keras
    ├── logs/                (TensorBoard)
    ├── curves_epoch[n].png  (plots intermédiaires)
    └── report_visuals.png   (graphiques rapport)
  Global : runs/all_runs_summary.csv



wandb: uploading history steps 61-61, summary, console lines 767-788
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
wandb:               lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_loss █▇▆▄▄▃▂▂▂▁▁▁▂▁▂▂▂▂▁▂▁▁▁▂▂▁▁▂▁▁▁▂▁▂▂▂▂▂▂▁
wandb: train_policy_acc █▅▃▁▄▃▃▄▃▃▇▄▄▄▃▃▇▃▆▆▃▅▄▃▁▅▄▄▄▂▂▁▂▂▃▄▂▅▄▅
wandb:  train_value_mae █▆▇▅▆▄▇▆▄▃▃▃▇▁▅▄▃▇▄▃▃▅▆▁▄▄▄▄▄▇▇▆█▁▄█▅▆▅▅
wandb:         val_loss █▆▅▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_policy_acc ▅▅▅▄▄▂▂▁▃▁▆▅▃▃▅▃▃▃▃▃▅▅▅▅▅▆▃▃▃█▁▃▅███▅██▄
wandb: 
wandb: Run summary:
wandb:            epoch 62
wandb:               lr 0.001
wandb:       train_loss 6.0006
wandb: train_policy_acc 0.0034
wandb:  train_value_mae 0.2917
wandb:         val_loss 6.00064
wandb:   val_policy_acc 0.0031
wandb: 
wandb: 🚀 View run model10_cnn_transformer_lr0.001_seed42_20260329_185524 at: https://wandb.ai/rouibiamine7-universit-paris-dauphine-psl/go-19x19/runs/pfz44u8m
wandb: ⭐️ View project at: https://wandb

In [15]:
import shutil
shutil.make_archive(
    '/kaggle/working/runs_export',
    'zip',
    '/kaggle/working',
    'runs'
)
print("\u2705 runs_export.zip prêt à télécharger depuis Output")


✅ runs_export.zip prêt à télécharger depuis Output
